# Util embedding experiments
Flow: load stimuli -> get embeddings -> fit a direction per subset -> compare directions.

In [19]:
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

HERE = Path.cwd()
sys.path.insert(0, str(HERE))

import datasets
import embed
import os
import pandas as pd

In [24]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
from pathlib import Path
import sys

In [43]:
ROOT_DIR = os.getcwd() + '/../../'
sys.path.append(ROOT_DIR)

sys.path.append(ROOT_DIR+'src/')
print(ROOT_DIR)

DATASET_DIRECTIONS_DIR = HERE / "dataset_direction_vectors/"
DATASET_DIRECTIONS_DIR

/Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/utility_experiments/util_embedding_experiments/../../


PosixPath('/Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/utility_experiments/util_embedding_experiments/dataset_direction_vectors')

In [29]:
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
import src.moral_projection as moral_projection
import src.generic_analysis_utils as generic_analysis_utils
import src.embedding_utils as embedding_utils
import src.core_process as core_process

importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)
importlib.reload(prompts)
importlib.reload(node)
importlib.reload(utils)
importlib.reload(core_process)



<module 'src.core_process' from '/Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/utility_experiments/util_embedding_experiments/../../src/core_process.py'>

## 1. Load stimuli
`SUBSETS` maps each subset label -> {"texts": list[str], "y": float array} for the directions we fit.

In [3]:
SUBSETS = {}  # label -> {"texts": list[str], "y": np.ndarray(float)}
# Small datasets first (fast feedback); large ETHICS train splits last.

# franken
exp1, exp2 = datasets.load_franken()
SUBSETS["franken-valence"] = {"texts": exp1["target"].tolist(), "y": exp1["avg_likert_rating"].to_numpy(float)}
SUBSETS["franken-good_vs_harm"] = {"texts": exp1["target"].tolist(), "y": (exp1["type"] == "good").to_numpy(float)}
SUBSETS["franken-severe_vs_mild"] = {"texts": exp1["target"].tolist(), "y": (exp1["strength"] == "severe").to_numpy(float)}
SUBSETS["franken-permissibility"] = {"texts": exp2["text"].tolist(), "y": exp2["avg_permissibility_rating"].to_numpy(float)}
SUBSETS["franken-intention"] = {"texts": exp2["text"].tolist(), "y": exp2["avg_intention_rating"].to_numpy(float)}

# nie / MoCa
nie = datasets.load_nie()
SUBSETS["nie-acceptability"] = {"texts": nie["text"].tolist(), "y": nie["p_yes"].to_numpy(float)}
for col, pos in [("causal_role", "Means"), ("personal_force", "Personal"),
                 ("evitability", "Inevitable"), ("beneficiary", "Other-beneficial")]:
    sub = nie.dropna(subset=[col])
    if sub[col].nunique() > 1:
        SUBSETS[f"nie-{col}"] = {"texts": sub["text"].tolist(), "y": (sub[col] == pos).to_numpy(float)}

# AITA
aita = datasets.load_aita()
SUBSETS["AITA-utility"] = {"texts": aita["outcome"].tolist(), "y": aita["utility"].to_numpy(float)}

# Holmes-Rahe
hr = datasets.load_holmes_rahe()
SUBSETS["Holmes-Rahe"] = {"texts": hr["event"].tolist(), "y": hr["lcu"].to_numpy(float)}

# GBD
gbd = datasets.load_gbd()
SUBSETS["GBD"] = {"texts": gbd["lay_description"].tolist(), "y": gbd["weight"].to_numpy(float)}

# ETHICS (large train splits - slowest to embed)
ethics = datasets.load_ethics()
a, b = ethics["utilitarianism"]["train"]["more_pleasant"], ethics["utilitarianism"]["train"]["less_pleasant"]
SUBSETS["ETHICS-util"] = {"texts": list(a) + list(b), "y": np.r_[np.ones(len(a)), np.zeros(len(b))]}

cm = ethics["commonsense"]["train"]
cm_short = cm[cm["is_short"]]
SUBSETS["ETHICS-cm"] = {"texts": cm_short["input"].tolist(), "y": cm_short["label"].to_numpy(float)}

deon = ethics["deontology"]["train"]
deon_text = (deon["scenario"] + " " + deon["excuse"]).tolist()
SUBSETS["ETHICS-deon"] = {"texts": deon_text, "y": deon["label"].to_numpy(float)}

for label, sub in SUBSETS.items():
    print(f"{label:24s} n={len(sub['texts']):5d}")

franken-valence          n=   80
franken-good_vs_harm     n=   80
franken-severe_vs_mild   n=   80
franken-permissibility   n=   80
franken-intention        n=   80
nie-acceptability        n=   44
nie-causal_role          n=   35
nie-personal_force       n=   35
nie-evitability          n=   35
nie-beneficiary          n=   35
AITA-utility             n=   59
Holmes-Rahe              n=   43
GBD                      n=  203
ETHICS-util              n=27476
ETHICS-cm                n= 6661
ETHICS-deon              n=18164


## 2. Get embeddings
Switch `MODEL` to any name in `embed.OPENAI_MODELS` or `embed.QWEN_MODELS` to change model.

In [ ]:
import os
MODEL = "Qwen/Qwen3-Embedding-4B"  # or any of embed.OPENAI_MODELS / embed.QWEN_MODELS
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")  # only needed if MODEL is an OpenAI model

embeddings = {}
for label, sub in SUBSETS.items():
    emb = embed.embed_dataset(sub["texts"], dataset=label, model=MODEL, api_key=OPENAI_API_KEY)
    X = np.array([emb[str(t)] for t in sub["texts"]])
    embeddings[label] = X
    print(f"{label:24s} {X.shape}")

franken-valence          (80, 3072)
franken-good_vs_harm     (80, 3072)
franken-severe_vs_mild   (80, 3072)
franken-permissibility   (80, 3072)
franken-intention        (80, 3072)
[text-embedding-3-large] embedding 44 texts for 'nie-acceptability'
[text-embedding-3-large] done, cache=42
nie-acceptability        (44, 3072)
[text-embedding-3-large] embedding 35 texts for 'nie-causal_role'
[text-embedding-3-large] done, cache=33
nie-causal_role          (35, 3072)
[text-embedding-3-large] embedding 35 texts for 'nie-personal_force'
[text-embedding-3-large] done, cache=33
nie-personal_force       (35, 3072)
[text-embedding-3-large] embedding 35 texts for 'nie-evitability'
[text-embedding-3-large] done, cache=33
nie-evitability          (35, 3072)
[text-embedding-3-large] embedding 35 texts for 'nie-beneficiary'
[text-embedding-3-large] done, cache=33
nie-beneficiary          (35, 3072)
[text-embedding-3-large] embedding 59 texts for 'AITA-utility'
[text-embedding-3-large] done, cache=59
AI

KeyboardInterrupt: 

In [4]:
MODEL = "text-embedding-3-large"  # or any of embed.OPENAI_MODELS / embed.QWEN_MODELS
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")  # only needed if MODEL is an OpenAI model

DATASETS_TO_EMBED = ["franken-valence", "franken-good_vs_harm", "franken-severe_vs_mild"] #select which datasets to embed and fit directions for

embeddings = {}
for label in DATASETS_TO_EMBED:
    sub = SUBSETS[label]
    emb = embed.embed_dataset(sub["texts"], dataset=label, model=MODEL, api_key=OPENAI_API_KEY)
    X = np.array([emb[str(t)] for t in sub["texts"]])
    embeddings[label] = X
    print(f"{label:24s} {X.shape}")

franken-valence          (80, 3072)
franken-good_vs_harm     (80, 3072)
franken-severe_vs_mild   (80, 3072)


## 3. Fit directions
Two simple methods to compare: ridge regression of the embeddings on `y`, and a plain
mean-difference between the high and low halves (split at the median for continuous `y`).

In [44]:
def ridge_dir(X, y, alpha=50.0):
    Xc = X - X.mean(0)
    w = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(Xc.shape[1]), Xc.T @ (y - y.mean()))
    return w / (np.linalg.norm(w) + 1e-9)

def meandiff_dir(X, y):
    classes = np.unique(y)
    if len(classes) == 2:            # binary: split by class membership, not by a threshold
        hi, lo = X[y == classes[1]].mean(0), X[y == classes[0]].mean(0)
    else:                            # continuous: split at the median
        thresh = np.median(y)
        hi, lo = X[y >= thresh].mean(0), X[y < thresh].mean(0)
    d = hi - lo
    return d / (np.linalg.norm(d) + 1e-9)

METHOD = "ridge"  # or "meandiff"

directions = {}
#for label, sub in SUBSETS.items():
#    X = embeddings[label]
#    d = ridge_dir(X, sub["y"]) if METHOD == "ridge" else meandiff_dir(X, sub["y"])
#    directions[label] = d

for label in DATASETS_TO_EMBED:
    sub = SUBSETS[label]
    X = embeddings[label]
    d = ridge_dir(X, sub["y"]) if METHOD == "ridge" else meandiff_dir(X, sub["y"])
    directions[label] = d

for dataset_label in directions.keys():
    direction_vector = directions[dataset_label]
    #np.save(f"{dataset_label}_direction.npy", direction_vector)
    np.save(os.path.join(DATASET_DIRECTIONS_DIR, f"{dataset_label}_direction.npy"), direction_vector)
    print(f"Saved direction vector for {dataset_label} to {dataset_label}_direction.npy")

Saved direction vector for franken-valence to franken-valence_direction.npy
Saved direction vector for franken-good_vs_harm to franken-good_vs_harm_direction.npy
Saved direction vector for franken-severe_vs_mild to franken-severe_vs_mild_direction.npy


## 4. Correlation between y and projection
For each subset, project its own embeddings onto its fitted direction and correlate
with the true `y`. In-sample (not held-out) - a quick sanity check that the direction
actually points the way it should, not a decodability estimate.

In [6]:
labels_c, rs = [], []
for label in DATASETS_TO_EMBED:
    sub = SUBSETS[label]
    proj = embeddings[label] @ directions[label]
    rs.append(np.corrcoef(proj, sub["y"])[0, 1])
    labels_c.append(label)

fig = go.Figure(go.Bar(x=labels_c, y=rs, text=rs, texttemplate="%{text:.2f}", textposition="outside"))
fig.update_layout(title="In-sample correlation between y and projection onto fitted direction",
                   yaxis_title="Pearson r", xaxis_tickangle=-60, width=900, height=500)
fig.show()

In [7]:
labels_c, rs = [], []
for label, sub in SUBSETS.items():
    proj = embeddings[label] @ directions[label]
    rs.append(np.corrcoef(proj, sub["y"])[0, 1])
    labels_c.append(label)

fig = go.Figure(go.Bar(x=labels_c, y=rs, text=rs, texttemplate="%{text:.2f}", textposition="outside"))
fig.update_layout(title="In-sample correlation between y and projection onto fitted direction",
                   yaxis_title="Pearson r", xaxis_tickangle=-60, width=900, height=500)
fig.show()

KeyError: 'franken-permissibility'

## 5. Cosine similarity between directions

In [6]:
labels = list(directions.keys())
D = np.array([directions[l] for l in labels])
C = D @ D.T

fig = px.imshow(C, x=labels, y=labels, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto")
fig.update_layout(title="Direction x direction cosine similarity", width=900, height=850)
fig.show()

## 6. Correlations between labels and directions (Cross-Dataset)
Like section 4, but cross-dataset: for every (labels_i, direction_j) pair, project
subset i's embeddings onto subset j's fitted direction and correlate with subset i's
own `y`. Row = whose data/labels, column = whose direction. The diagonal reproduces
section 4's numbers; off-diagonal cells show how well one dataset's direction
generalizes to another's labels (e.g. franken-valence data projected onto the
nie-causal_role direction, correlated with the franken-valence labels).

In [7]:
Cross = np.zeros((len(labels), len(labels)))
for i, li in enumerate(labels):
    Xi, yi = embeddings[li], SUBSETS[li]["y"]
    for j, lj in enumerate(labels):
        proj = Xi @ directions[lj]
        Cross[i, j] = np.corrcoef(proj, yi)[0, 1]

fig = px.imshow(Cross, x=labels, y=labels, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto")
fig.update_layout(title="Correlation between y and projection, cross-dataset (row = data, column = direction)",
                   width=900, height=850)
fig.update_xaxes(title="direction")
fig.update_yaxes(title="data / labels")
fig.show()